# Depth Up-Res Training Pipeline

End-to-end pipeline: postprocess → dataset assembly → training → evaluation.

| Step | Section | What it does |
|------|---------|-------------|
| 1 | Postprocess | Landmark-align + normalize 1024×1024 rendered maps |
| 2 | Dataset | 2.1 Reorganize → 2.2 Face masks → 2.3 LR/HR pairs → 2.4 Viz |
| 3 | Training | 3.1 Main 8-bit → 3.2 16-bit ablation → 3.3–3.4 Normal ablations → 3.5 DSINE |
| 4 | Evaluation | Curves, visual comparison, baselines table, mesh quality |

Every section has a skip-guard — safe to re-run without repeating finished steps.


In [ ]:
# === Setup ===
import sys, os, time, json
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image

# Make scripts/ importable
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'FaceLift':
    PROJECT_ROOT = PROJECT_ROOT.parent
SCRIPTS = PROJECT_ROOT / 'scripts'
sys.path.insert(0, str(SCRIPTS))
os.chdir(PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')
print(f'CWD: {Path.cwd()}')

## Step 1. Postprocess 1024×1024 maps

Runs the full postprocessing pipeline: landmark alignment, nose-relative depth
normalization, bilateral normal smoothing, hole filling, depth↔normal consistency check.
Output: `data/postprocessed/{rgb,depth,normal,opacity}/`.


In [ ]:
# === Run postprocess (auto-skip if already done) ===
import importlib
import postprocess_maps
importlib.reload(postprocess_maps)
from postprocess_maps import LandmarkAligner, postprocess_single

src = {
    'cropped': Path('data/cropped_faces'),
    'rgb':     Path('data/rgb_rendered'),
    'depth':   Path('data/depth_maps'),
    'normal':  Path('data/normal_maps'),
    'opacity': Path('data/opacity_maps'),
}
out_dirs = {
    'rgb':     Path('data/postprocessed/rgb'),
    'depth':   Path('data/postprocessed/depth'),
    'normal':  Path('data/postprocessed/normal'),
    'opacity': Path('data/postprocessed/opacity'),
}
for d in out_dirs.values():
    d.mkdir(parents=True, exist_ok=True)

cropped_stems = {p.stem for p in src['cropped'].glob('*.png')}
rendered_stems = {p.stem for p in src['rgb'].glob('*.png')}
samples = sorted(cropped_stems & rendered_stems)

# ---- SKIP GUARD ----
existing_depth = {p.stem for p in out_dirs['depth'].glob('*.png')}
missing = set(samples) - existing_depth
if samples and not missing:
    print(f'[SKIP] postprocess already done: {len(existing_depth)}/{len(samples)} outputs exist in {out_dirs["depth"]}')
    print('       Delete data/postprocessed/ or edit this cell to re-run')
else:
    if existing_depth:
        print(f'[RESUME] {len(existing_depth)}/{len(samples)} already done, processing {len(missing)} missing')
        samples = sorted(missing)
    print(f'Samples to process: {len(samples)}')
    sample_rgb = Image.open(src['rgb'] / f'{samples[0]}.png')
    sample_depth = Image.open(src['depth'] / f'{samples[0]}.png')
    print(f"  RGB: {sample_rgb.size} {sample_rgb.mode}")
    print(f"  Depth: {sample_depth.size} {sample_depth.mode}")

    aligner = LandmarkAligner()
    pp_config = {
        'opacity_threshold': 0.3, 'min_component_area': 100,
        'use_ecc_refine': True, 'fill_holes': True, 'max_hole_area': 5000,
        'normals_from_depth': True, 'camera_hfov': 50.0,
        'bilateral_d': 9, 'bilateral_sigma_color': 30.0, 'bilateral_sigma_space': 30.0,
    }

    n_ok, n_fail = 0, 0
    scores = []
    t0 = time.time()
    for i, name in enumerate(samples):
        try:
            r = postprocess_single(
                str(src['cropped']/f'{name}.png'),
                str(src['rgb']/f'{name}.png'),
                str(src['depth']/f'{name}.png'),
                str(src['normal']/f'{name}.png'),
                str(src['opacity']/f'{name}.png'),
                aligner, pp_config,
            )
            Image.fromarray(r['render_rgb']).save(out_dirs['rgb']/f'{name}.png')
            depth16 = (np.clip(r['depth'],0,1)*65535).astype(np.uint16)
            Image.fromarray(depth16, mode='I;16').save(out_dirs['depth']/f'{name}.png')
            Image.fromarray(r['normal']).save(out_dirs['normal']/f'{name}.png')
            Image.fromarray(r['opacity'], mode='L').save(out_dirs['opacity']/f'{name}.png')
            scores.append(r.get('consistency_score', 0.0))
            n_ok += 1
        except Exception as e:
            print(f'FAIL [{name}]: {e}')
            n_fail += 1
        if (i+1) % 50 == 0 or (i+1) == len(samples):
            elapsed = time.time() - t0
            eta = elapsed/(i+1) * (len(samples)-i-1)
            print(f'  [{i+1}/{len(samples)}] elapsed={elapsed:.0f}s ETA={eta:.0f}s')
    print(f"\nDone. ok={n_ok} fail={n_fail}")
    if scores:
        cs = np.array(scores)
        print(f'Consistency: mean={cs.mean():.3f} min={cs.min():.3f} max={cs.max():.3f}')
        print(f'  >0.9: {(cs>0.9).sum()}/{len(cs)} ({(cs>0.9).mean()*100:.0f}%)')

# Verify
out_sample = sorted(out_dirs['depth'].glob('*.png'))[0]
print(f'\nOutput depth size: {Image.open(out_sample).size}')


### 1.1 Visualize postprocessed results


In [ ]:
# === Visualize postprocessed samples ===
import random
random.seed(0)
viz_samples = random.sample(sorted(out_dirs['rgb'].glob('*.png')), min(4, len(samples)))

fig, axes = plt.subplots(len(viz_samples), 5, figsize=(20, 4*len(viz_samples)))
if len(viz_samples) == 1:
    axes = axes[None,:]

for row, p in enumerate(viz_samples):
    name = p.stem
    orig = np.array(Image.open(src['cropped']/f'{name}.png').convert('RGB'))
    rgb = np.array(Image.open(out_dirs['rgb']/f'{name}.png').convert('RGB'))
    depth = np.array(Image.open(out_dirs['depth']/f'{name}.png'), dtype=np.float32) / 65535.0
    normal = np.array(Image.open(out_dirs['normal']/f'{name}.png').convert('RGB'))
    opacity = np.array(Image.open(out_dirs['opacity']/f'{name}.png').convert('L'))

    axes[row,0].imshow(orig); axes[row,0].set_title(f'Original\n{name}')
    axes[row,1].imshow(rgb); axes[row,1].set_title('Rendered RGB (1024)')
    axes[row,2].imshow(depth, cmap='inferno', vmin=0, vmax=1); axes[row,2].set_title(f'Depth (1024, max={depth.max():.2f})')
    axes[row,3].imshow(normal); axes[row,3].set_title('Normal (1024)')
    axes[row,4].imshow(opacity, cmap='gray'); axes[row,4].set_title('Opacity')
    for ax in axes[row]:
        ax.axis('off')

plt.tight_layout()
plt.show()

## Step 2. Dataset assembly

Runs linearly 2.1 → 2.4 with skip-guards at each step.

### 2.1 Reorganize into train/val split (90/10, seed=42)


In [ ]:
# === Reorganize dataset (auto-skip if already done) ===
import subprocess

ds_train = Path('data/dataset/train/depth')
ds_val   = Path('data/dataset/val/depth')

if ds_train.exists() and ds_val.exists() and any(ds_train.glob('*.png')) and any(ds_val.glob('*.png')):
    print(f'[SKIP] dataset already reorganized')
else:
    result = subprocess.run([sys.executable, 'scripts/reorganize_dataset.py'],
                           capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr)

# Verify
print(f"\ntrain: {len(list(ds_train.glob('*.png')))} samples")
print(f"val: {len(list(ds_val.glob('*.png')))} samples")
print(f"sample depth size: {Image.open(sorted(ds_train.glob('*.png'))[0]).size}")


### 2.2 Build face masks

Morphology-based masks from `cropped_faces` (white background → non-white foreground).
Produces `dataset/{train,val}/{mask,mask_lr}/*.png`.
DepthUpResDataset picks these up automatically for mask-aware loss.
Falls back to all-ones mask if masks are missing.


In [ ]:
# === Build face masks (auto-skip if already done) ===
import subprocess

train_mask    = Path('data/dataset/train/mask')
val_mask      = Path('data/dataset/val/mask')
train_mask_lr = Path('data/dataset/train/mask_lr')
val_mask_lr   = Path('data/dataset/val/mask_lr')

def _count(d):
    return len(list(d.glob('*.png'))) if d.exists() else 0

train_depth_n = _count(Path('data/dataset/train/depth'))
val_depth_n   = _count(Path('data/dataset/val/depth'))

all_present = all(
    _count(d) == n and n > 0
    for d, n in [
        (train_mask,    train_depth_n),
        (train_mask_lr, train_depth_n),
        (val_mask,      val_depth_n),
        (val_mask_lr,   val_depth_n),
    ]
)

if all_present:
    print('[SKIP] face masks already present for every sample')
    print(f'  train mask={_count(train_mask)}  mask_lr={_count(train_mask_lr)}  '
          f'(expected {train_depth_n})')
    print(f'  val   mask={_count(val_mask)}  mask_lr={_count(val_mask_lr)}  '
          f'(expected {val_depth_n})')
else:
    result = subprocess.run(
        [sys.executable, 'scripts/make_face_masks.py'],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr)
        raise RuntimeError(f'make_face_masks.py failed: exit code {result.returncode}')
    # Verify
    for d, n in [(train_mask, train_depth_n), (val_mask, val_depth_n),
                 (train_mask_lr, train_depth_n), (val_mask_lr, val_depth_n)]:
        got = _count(d)
        tag = 'OK' if got >= n - 5 else 'WARN'  # allow up to 5 missing (no cropped src)
        print(f'  [{tag}] {d}: {got} / {n}')


### 2.2b Visualize: face mask vs opacity mask

Confirms that hallucinated shoulders (from FaceLift's 3DGS opacity) are
excluded by the tighter face mask. The face mask should be a strict subset
of the opacity mask.


In [ ]:
# === Viz: face mask vs opacity (see that hallucinated shoulders are excluded) ===
import random
random.seed(0)

cropped_dir = Path('data/cropped_faces')
rgb_dir     = Path('data/postprocessed/rgb')
op_dir      = Path('data/postprocessed/opacity')
fm_dir      = Path('data/dataset/train/mask')

shared = sorted(
    set(p.stem for p in fm_dir.glob('*.png'))
    & set(p.stem for p in rgb_dir.glob('*.png'))
    & set(p.stem for p in op_dir.glob('*.png'))
    & set(p.stem for p in cropped_dir.glob('*.png'))
)

if not shared:
    print('No overlap between cropped_faces / rgb_rendered / opacity / face mask — '
          'run section 2.2 first.')
else:
    picks = random.sample(shared, min(3, len(shared)))
    fig, axes = plt.subplots(len(picks), 5, figsize=(20, 4 * len(picks)))
    if len(picks) == 1:
        axes = axes.reshape(1, -1)

    for row, stem in enumerate(picks):
        cropped = np.array(Image.open(cropped_dir / f'{stem}.png').convert('RGB'))
        rgb_r   = np.array(Image.open(rgb_dir     / f'{stem}.png').convert('RGB'))
        op      = np.array(Image.open(op_dir      / f'{stem}.png').convert('L'))
        fm      = np.array(Image.open(fm_dir      / f'{stem}.png').convert('L'))

        # Binarize
        op_b = op > 127
        fm_b = fm > 127

        # Difference: in opacity but NOT in face mask = hallucinated region
        hallu = op_b & ~fm_b

        # Masked RGB for visual sanity
        rgb_fm = rgb_r.copy()
        rgb_fm[~fm_b] = 255   # paint non-mask white

        axes[row, 0].imshow(cropped);    axes[row, 0].set_title(f'{stem}\ncropped_faces (source)', fontsize=9)
        axes[row, 1].imshow(rgb_r);      axes[row, 1].set_title('rendered RGB\n(incl. hallucinated shoulders)', fontsize=9)
        axes[row, 2].imshow(op_b, cmap='gray'); axes[row, 2].set_title(
            f'opacity mask\nfg={op_b.mean()*100:.1f}%  (includes hallucination)', fontsize=9)
        axes[row, 3].imshow(fm_b, cmap='gray'); axes[row, 3].set_title(
            f'face mask (2.2)\nfg={fm_b.mean()*100:.1f}%  (face only, no hallucination)', fontsize=9)
        axes[row, 4].imshow(hallu, cmap='Reds'); axes[row, 4].set_title(
            f'opacity \\ face_mask\n= hallucinated region {hallu.mean()*100:.1f}%', fontsize=9)
        for j in range(5):
            axes[row, j].set_axis_off()

    plt.tight_layout()
    plt.show()
    print(f'Showing {len(picks)}  samples. Col 4 should be smaller than col 3; col 5 shows the excluded hallucinated region (collar/shirt/background).')


### 2.3 Generate LR/HR pairs (256×256, 8-bit + 16-bit)


In [ ]:
# === Make LR/HR pairs (auto-skip if already done) ===
import subprocess

lr_dirs = [
    Path('data/dataset/train/depth_lr_8bit'),
    Path('data/dataset/train/depth_lr_16bit'),
    Path('data/dataset/val/depth_lr_8bit'),
    Path('data/dataset/val/depth_lr_16bit'),
]
all_present = all(d.exists() and any(d.glob('*.png')) for d in lr_dirs)

if all_present:
    counts = {d.name + '/' + d.parent.name: len(list(d.glob('*.png'))) for d in lr_dirs}
    print('[SKIP] LR/HR pairs already generated')
    for k, v in counts.items():
        print(f'  {k}: {v}')
else:
    result = subprocess.run([sys.executable, 'scripts/make_lr_hr_pairs.py'],
                           capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr)


### 2.4 Visualize LR/HR pairs


In [ ]:
# === Visualize LR/HR pairs ===
# Full-frame 3×3 grid + zoom-in 3×3 grid to show pixel-level degradation
import random
random.seed(0)   # reproducible sample pick

hr_dir   = Path('data/dataset/train/depth')
lr8_dir  = Path('data/dataset/train/depth_lr_8bit')
lr16_dir = Path('data/dataset/train/depth_lr_16bit')

samples_viz = random.sample(sorted(hr_dir.glob('*.png')), 3)

# ---------- Top panel: full-frame 256 vs 256 vs 1024 ----------
fig, axes = plt.subplots(3, 3, figsize=(15, 15))
for row, p in enumerate(samples_viz):
    name = p.name
    hr   = np.array(Image.open(hr_dir   / name), dtype=np.float32) / 65535.0
    lr16 = np.array(Image.open(lr16_dir / name), dtype=np.float32) / 65535.0
    lr8  = np.array(Image.open(lr8_dir  / name), dtype=np.float32) / 255.0

    axes[row, 0].imshow(lr8,  cmap='inferno', vmin=0, vmax=1)
    axes[row, 0].set_title(f'LR 256 8-bit\n{p.stem}', fontsize=9)
    axes[row, 1].imshow(lr16, cmap='inferno', vmin=0, vmax=1)
    axes[row, 1].set_title('LR 256 16-bit', fontsize=9)
    axes[row, 2].imshow(hr,   cmap='inferno', vmin=0, vmax=1)
    axes[row, 2].set_title('HR 1024 16-bit', fontsize=9)
    for ax in axes[row]: ax.axis('off')
plt.suptitle('Full-frame view — looks similar because matplotlib auto-scales them', y=1.01, fontsize=10)
plt.tight_layout()
plt.show()

# ---------- Bottom panel: zoom-in (center-crop nose region, shown at same display size) ----------
# HR crop: 256×256 centered = pixel range [384:640, 384:640]
# LR crop: 64×64 centered   = pixel range [96:160, 96:160]
# imshow interpolation='nearest' keeps pixels raw → 8-bit pixelation becomes obvious
HR_SZ, LR_SZ = 1024, 256
HR_CROP = 256   # center 256×256 of HR (about nose/mouth region)
LR_CROP = HR_CROP // (HR_SZ // LR_SZ)   # = 64

hr_lo = (HR_SZ - HR_CROP) // 2
hr_hi = hr_lo + HR_CROP
lr_lo = (LR_SZ - LR_CROP) // 2
lr_hi = lr_lo + LR_CROP

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
for row, p in enumerate(samples_viz):
    name = p.name
    hr   = np.array(Image.open(hr_dir   / name), dtype=np.float32) / 65535.0
    lr16 = np.array(Image.open(lr16_dir / name), dtype=np.float32) / 65535.0
    lr8  = np.array(Image.open(lr8_dir  / name), dtype=np.float32) / 255.0

    lr8_crop  = lr8[lr_lo:lr_hi, lr_lo:lr_hi]
    lr16_crop = lr16[lr_lo:lr_hi, lr_lo:lr_hi]
    hr_crop   = hr[hr_lo:hr_hi, hr_lo:hr_hi]

    # interpolation='nearest' preserves pixel staircasing — no smoothing by matplotlib
    axes[row, 0].imshow(lr8_crop,  cmap='inferno', vmin=0, vmax=1, interpolation='nearest')
    axes[row, 0].set_title(f'LR 8-bit crop  {LR_CROP}×{LR_CROP}\n{p.stem}  (visible pixelation + quantization banding)', fontsize=9)
    axes[row, 1].imshow(lr16_crop, cmap='inferno', vmin=0, vmax=1, interpolation='nearest')
    axes[row, 1].set_title(f'LR 16-bit crop {LR_CROP}×{LR_CROP}\n(same resolution, much less banding)', fontsize=9)
    axes[row, 2].imshow(hr_crop,   cmap='inferno', vmin=0, vmax=1, interpolation='nearest')
    axes[row, 2].set_title(f'HR 16-bit crop  {HR_CROP}×{HR_CROP}\n(4x pixel density + full precision)', fontsize=9)
    for ax in axes[row]: ax.axis('off')
plt.suptitle(
    f'Center crop  (HR {HR_CROP}×{HR_CROP} vs LR {LR_CROP}×{LR_CROP})  '
    f'— interpolation=nearest so 8-bit quantization + spatial pixelation is visible',
    y=1.01, fontsize=10,
)
plt.tight_layout()
plt.show()

# ---------- (Optional) quick numerical summary: depth std inside the crop ----------
# 8-bit quantization causes staircase artifacts in smooth regions; visible as high local std bins
for p in samples_viz:
    name = p.name
    hr   = np.array(Image.open(hr_dir   / name), dtype=np.float32) / 65535.0
    lr16 = np.array(Image.open(lr16_dir / name), dtype=np.float32) / 65535.0
    lr8  = np.array(Image.open(lr8_dir  / name), dtype=np.float32) / 255.0
    # Unique value count inside the crop — 8-bit typically ≤ 64, 16-bit usually > 1000
    n8  = len(np.unique((lr8  [lr_lo:lr_hi, lr_lo:lr_hi] * 255   ).astype(np.int32)))
    n16 = len(np.unique((lr16 [lr_lo:lr_hi, lr_lo:lr_hi] * 65535).astype(np.int32)))
    nhr = len(np.unique((hr   [hr_lo:hr_hi, hr_lo:hr_hi] * 65535).astype(np.int32)))
    print(f'  {p.stem}: unique depth values in crop — '
          f'LR-8bit={n8:4d}  LR-16bit={n16:5d}  HR={nhr:5d}')


## Step 3. Training

### 3.1 Main run (8-bit LR, depth-only)

Primary setting. Bicubic residual learning + mask-aware L1 + gradient loss.
Config: `lr_input=8bit`, `base_ch=32`, batch=2, grad_accum=4, AMP.
Checkpoint: `checkpoints/depth_upres_8bit_ch32/`.


In [ ]:
# === Training (parameterized — supports 8bit/16bit ablation, separate checkpoints) ===
# PATCH v2: defaults bumped to base_ch=64 / epochs=150 for the final main run.
import subprocess

# -------- Toggle: change this line to switch input type --------
LR_KIND = '8bit'     # '8bit' or '16bit' — switch to '16bit' for ablation run
EPOCHS  = 100
BASE_CH = 32         # 32=baseline, 64=larger model (auto-switches to batch=1 grad_accum=8)
# ----------------------------------------------

ckpt_dir  = Path(f'checkpoints/depth_upres_{LR_KIND}_ch{BASE_CH}')
best_ckpt = ckpt_dir / 'best.pt'
train_log = ckpt_dir / 'train_log.json'

if best_ckpt.exists():
    print(f'[SKIP] Found trained model: {best_ckpt}')
    if train_log.exists():
        import json as _json
        hist = _json.loads(train_log.read_text())
        if hist:
            best = min(hist, key=lambda r: r['val_loss'])
            print(f"       Trained {len(hist)}  epochs, best val={best['val_loss']:.5f} @ epoch {best['epoch']}")
            print(f"       best val_l1 = {best.get('val_l1', 'n/a')}")
    print(f'       To retrain, first delete: {best_ckpt}')
    print(f'       To change input type, edit LR_KIND (auto-writes to separate checkpoint dir)')
else:
    # Auto-adjust batch size based on base_ch to avoid 8GB VRAM OOM
    # base_ch=64 has ~4x params; at 1024x1024, must use batch=1; grad_accum=8 keeps effective batch=8
    # On 8GB 4070 Laptop with Windows + other GPU users (browser / desktop),
    # batch=1 grad_accum=8 is the only setting that reliably fits.
    # Effective batch = 8 either way.
    batch_size, grad_accum = 1, 8

    cmd = [
        sys.executable, 'scripts/train_depth_upres.py',
        '--lr_input', LR_KIND,
        '--epochs', str(EPOCHS),
        '--batch_size', str(batch_size),
        '--grad_accum', str(grad_accum),
        '--base_ch', str(BASE_CH),
        '--checkpoints', str(ckpt_dir),
        '--num_workers', '0',   # Windows cv2+multiproc OOM safe
    ]
    print(f'>>> Training: LR_KIND={LR_KIND}, base_ch={BASE_CH}, epochs={EPOCHS}')
    print(f'>>> Checkpoint dir: {ckpt_dir}')
    print(f'>>> Effective batch = {batch_size * grad_accum}')
    print('Running:', ' '.join(cmd))
    print('-' * 60)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    print(f'\nExit code: {proc.returncode}')


### 3.2 Ablation: 16-bit LR (depth-only)

Quantifies how much difficulty comes from 8-bit quantization.
16-bit removes quantization noise → pure ×4 spatial SR.
Gap Δval_l1 = (3.1 − 3.2) measures the 8-bit penalty.
Checkpoint: `checkpoints/depth_upres_16bit_ch32/`.


In [ ]:
# === 16-bit LR ablation run (base_ch=64, epochs=150) ===
# Reuses scripts/train_depth_upres.py with --lr_input 16bit.
# If 8-bit hasn't finished, complete it first — can't run two training sessions simultaneously.
import subprocess

ABL_LR_KIND = '16bit'
ABL_EPOCHS  = 100
ABL_BASE_CH = 32

abl_ckpt_dir  = Path(f'checkpoints/depth_upres_{ABL_LR_KIND}_ch{ABL_BASE_CH}')
abl_best      = abl_ckpt_dir / 'best.pt'
abl_log       = abl_ckpt_dir / 'train_log.json'

if abl_best.exists():
    print(f'[SKIP] Found 16-bit ablation model: {abl_best}')
    if abl_log.exists():
        import json as _json
        hist = _json.loads(abl_log.read_text())
        if hist:
            best = min(hist, key=lambda r: r['val_loss'])
            print(f"       Trained {len(hist)}  epochs, best val={best['val_loss']:.5f} @ epoch {best['epoch']}")
    print('       To retrain, delete best.pt above')
else:
    # Same VRAM strategy as main run
    # On 8GB 4070 Laptop batch=1 grad_accum=8 is the safest setting.
    batch_size, grad_accum = 1, 8

    cmd = [
        sys.executable, 'scripts/train_depth_upres.py',
        '--lr_input', ABL_LR_KIND,
        '--epochs', str(ABL_EPOCHS),
        '--batch_size', str(batch_size),
        '--grad_accum', str(grad_accum),
        '--base_ch', str(ABL_BASE_CH),
        '--checkpoints', str(abl_ckpt_dir),
        '--num_workers', '0',   # Windows cv2+multiproc OOM safe
    ]
    print(f'>>> Ablation run: LR_KIND={ABL_LR_KIND}, base_ch={ABL_BASE_CH}, epochs={ABL_EPOCHS}')
    print(f'>>> Checkpoint dir: {abl_ckpt_dir}')
    print(f'>>> Effective batch = {batch_size * grad_accum}')
    print('Running:', ' '.join(cmd))
    print('-' * 60)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    print(f'\nAblation exit code: {proc.returncode}')


### 3.2b Preview: bilateral smoothing strength

Compare smoothing strengths before committing to a training run.
Three levels: Raw 3DGS | Medium (d=11, σC=40, σS=25) | Strong 2× (d=15, σC=75, σS=75).
Target: ~3–8° angular change on flat skin, <15° overall.


In [ ]:
# === Preview: bilateral smoothing on GT normals (3 strength levels) ===
# 4 columns: [cropped | raw normal | medium bilateral | strong bilateral x2]
# The "strong x2" is exactly what --smooth_normal applies at training time.
import cv2
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import random

random.seed(0)

cropped_dir = Path('data/cropped_faces')
normal_dir  = Path('data/dataset/train/normal')
stems = sorted(p.stem for p in normal_dir.glob('*.png'))
if not stems:
    print('No normals found at data/dataset/train/normal/')
else:
    preferred = ['ffhq_35368_face', 'ffhq_39217_face', 'ffhq_01362_face', 'ffhq_21435_face']
    picks = [s for s in preferred if s in stems]
    if len(picks) < 4:
        picks += random.sample([s for s in stems if s not in picks], 4 - len(picks))
    picks = picks[:4]
    
    fig, axes = plt.subplots(len(picks), 4, figsize=(16, 4 * len(picks)))
    if len(picks) == 1:
        axes = axes.reshape(1, -1)
    
    for row, stem in enumerate(picks):
        # 0. Cropped face
        cp = cropped_dir / f'{stem}.png'
        if cp.exists():
            face = np.array(Image.open(cp).convert('RGB'))
            axes[row, 0].imshow(face)
        axes[row, 0].set_title(f'{stem}\n(cropped face)', fontsize=9)
        axes[row, 0].axis('off')
        
        # 1. Raw normal
        n_raw = np.array(Image.open(normal_dir / f'{stem}.png').convert('RGB'))
        axes[row, 1].imshow(n_raw)
        axes[row, 1].set_title('Raw normal\n(3DGS render, stippled)', fontsize=9)
        axes[row, 1].axis('off')
        
        # 2. Medium: single pass, d=11 σC=40 σS=25 (weaker version tested previously)
        n_medium = cv2.bilateralFilter(n_raw, d=11, sigmaColor=40.0, sigmaSpace=25.0)
        axes[row, 2].imshow(n_medium)
        axes[row, 2].set_title('Medium smoothing\n(1×bilateral d=11 σC=40)', fontsize=9)
        axes[row, 2].axis('off')
        
        # 3. Strong: two passes, d=15 σC=75 σS=75 (used for training)
        n_strong = n_raw
        for _ in range(2):
            n_strong = cv2.bilateralFilter(n_strong, d=15, sigmaColor=75.0, sigmaSpace=75.0)
        axes[row, 3].imshow(n_strong)
        axes[row, 3].set_title('Strong smoothing (used by --smooth_normal)\n(2×bilateral d=15 σC=75 σS=75)',
                                fontsize=9)
        axes[row, 3].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Quantify
    print('\nAngular change measure (per-pixel mean, within opacity mask):')
    print(f"{'sample':<22s}  {'medium':>9s}  {'strong':>9s}")
    for stem in picks:
        n_raw_arr = np.array(Image.open(normal_dir / f'{stem}.png').convert('RGB'))
        n_med = cv2.bilateralFilter(n_raw_arr, d=11, sigmaColor=40.0, sigmaSpace=25.0)
        n_str = n_raw_arr
        for _ in range(2):
            n_str = cv2.bilateralFilter(n_str, d=15, sigmaColor=75.0, sigmaSpace=75.0)
        
        def angdiff(a, b):
            ra = a.astype(np.float32) / 255.0 * 2 - 1
            rb = b.astype(np.float32) / 255.0 * 2 - 1
            ra /= (np.linalg.norm(ra, axis=-1, keepdims=True) + 1e-8)
            rb /= (np.linalg.norm(rb, axis=-1, keepdims=True) + 1e-8)
            cos = np.clip((ra * rb).sum(axis=-1), -1, 1)
            return np.rad2deg(np.arccos(cos))
        
        ang_med = angdiff(n_raw_arr, n_med)
        ang_str = angdiff(n_raw_arr, n_str)
        opacity = np.array(Image.open(f'data/dataset/train/opacity/{stem}.png').convert('L'))
        mask = opacity > 127
        print(f'  {stem:<22s}  {ang_med[mask].mean():>8.1f}°  {ang_str[mask].mean():>8.1f}°')
    
    print()
    print('Compare column 3 vs column 4 to decide:')
    print('  - Pore/stipple gone AND features clear → use strong (training default)')
    print('  - Strong too aggressive (flat face / blurry nose bridge) → change range(2) to range(1) in training script')
    print('  - Medium is enough → revert script to single pass d=11 sigmaC=40 sigmaS=25')


### 3.3 Ablation: 8-bit + normal loss (3DGS rendered normals)

**Negative result / minor claim.** 3DGS-rendered normals have per-splat aliasing
that makes them harmful as supervision. val_l1 worsens from 0.00232 → 0.00370.
Confirmed by Normal-GS (NeurIPS 2024), 2DGS (SIGGRAPH 2024), SuGaR, DN-Splatter.

**Currently disabled** — this experiment is complete and the result is recorded.


In [ ]:
# === 3.3 Normal-aware ablation (8-bit LR + cosine loss) ===
# Change NA_NORMAL_WEIGHT and re-run to sweep weights; each weight writes to
# its own checkpoint dir so prior runs are preserved (for the ablation table).
import subprocess

NA_LR_KIND       = '8bit'
NA_EPOCHS        = 150
NA_BASE_CH       = 32
NA_NORMAL_WEIGHT = 0.05   # sweep: try 0.05 first; prior run at 0.2 archived separately

# One-time migration: old runs used `..._normal/` without weight in the name.
# If that legacy dir exists (from the w=0.2 run), rename it to `..._normal_w020/`
# so skip guards don't collide with new weight-suffixed runs.
_legacy = Path(f'checkpoints/depth_upres_{NA_LR_KIND}_ch{NA_BASE_CH}_normal')
_legacy_target = Path(f'checkpoints/depth_upres_{NA_LR_KIND}_ch{NA_BASE_CH}_normal_w020')
if _legacy.exists() and (_legacy / 'best.pt').exists() and not _legacy_target.exists():
    _legacy.rename(_legacy_target)
    print(f'[migrate] renamed {_legacy.name} -> {_legacy_target.name} (historical w=0.2 data)')

# Weight suffix for this run
_w_tag = f'w{int(round(NA_NORMAL_WEIGHT * 1000)):03d}_sm'   # _sm = bilateral-smoothed GT normal
na_ckpt_dir = Path(f'checkpoints/depth_upres_{NA_LR_KIND}_ch{NA_BASE_CH}_normal_{_w_tag}')
na_best     = na_ckpt_dir / 'best.pt'
na_log      = na_ckpt_dir / 'train_log.json'

if True:  # FORCE SKIP — 3DGS rendered normal proven failed, see §3.5 for DSINE fix
    print(f'[SKIP] already trained: {na_ckpt_dir}')
    if na_log.exists():
        hist = json.loads(na_log.read_text())
        if hist:
            best = min(hist, key=lambda r: r['val_loss'])
            print(f"       {len(hist)}  epochs, best val={best['val_loss']:.5f} "
                  f"@ epoch {best['epoch']}, val_l1={best.get('val_l1', 'n/a')}")
    print('       change NA_NORMAL_WEIGHT to sweep a different weight')
else:
    batch_size, grad_accum = 1, 8   # same as 3.1 / 3.2

    cmd = [
        sys.executable, 'scripts/train_depth_upres.py',
        '--lr_input', NA_LR_KIND,
        '--epochs', str(NA_EPOCHS),
        '--batch_size', str(batch_size),
        '--grad_accum', str(grad_accum),
        '--base_ch', str(NA_BASE_CH),
        '--checkpoints', str(na_ckpt_dir),
        '--num_workers', '0',
        '--normal_loss_weight', str(NA_NORMAL_WEIGHT),
        '--smooth_normal',
    ]
    print(f'>>> Normal-aware run: LR={NA_LR_KIND}, weight={NA_NORMAL_WEIGHT}')
    print(f'>>> Checkpoint       : {na_ckpt_dir}')
    print('Running:', ' '.join(cmd))
    print('-' * 60)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    print(f'\n3.3 exit code: {proc.returncode}')


### 3.4 Ablation: 16-bit + normal loss (3DGS rendered normals)

Pairs with §3.3. Proves 3DGS normals fail even without quantization noise.

**Currently disabled** — experiment complete.


In [ ]:
# === 3.4 Normal-aware ablation (16-bit LR + cosine loss) ===
# Same structure as 3.3: weight goes into the ckpt dir, sweep by changing the value.
import subprocess

NA16_LR_KIND       = '16bit'
NA16_EPOCHS        = 150
NA16_BASE_CH       = 32
NA16_NORMAL_WEIGHT = 0.05   # matches 3.3's default for direct 2x2 comparison

# One-time migration: move legacy `..._normal/` (broken w=0.2 run) out of the way
_legacy = Path(f'checkpoints/depth_upres_{NA16_LR_KIND}_ch{NA16_BASE_CH}_normal')
_legacy_target = Path(f'checkpoints/depth_upres_{NA16_LR_KIND}_ch{NA16_BASE_CH}_normal_w020')
if _legacy.exists() and (_legacy / 'best.pt').exists() and not _legacy_target.exists():
    _legacy.rename(_legacy_target)
    print(f'[migrate] renamed {_legacy.name} -> {_legacy_target.name} (w=0.2 run — note: may be broken)')

_w_tag = f'w{int(round(NA16_NORMAL_WEIGHT * 1000)):03d}_sm'
na16_ckpt_dir = Path(f'checkpoints/depth_upres_{NA16_LR_KIND}_ch{NA16_BASE_CH}_normal_{_w_tag}')
na16_best     = na16_ckpt_dir / 'best.pt'
na16_log      = na16_ckpt_dir / 'train_log.json'

if True:  # FORCE SKIP — 3DGS rendered normal proven failed, see §3.5 for DSINE fix
    print(f'[SKIP] already trained: {na16_ckpt_dir}')
    if na16_log.exists():
        hist = json.loads(na16_log.read_text())
        if hist:
            best = min(hist, key=lambda r: r['val_loss'])
            print(f"       {len(hist)}  epochs, best val={best['val_loss']:.5f} "
                  f"@ epoch {best['epoch']}, val_l1={best.get('val_l1', 'n/a')}")
else:
    batch_size, grad_accum = 1, 8
    cmd = [
        sys.executable, 'scripts/train_depth_upres.py',
        '--lr_input', NA16_LR_KIND,
        '--epochs', str(NA16_EPOCHS),
        '--batch_size', str(batch_size),
        '--grad_accum', str(grad_accum),
        '--base_ch', str(NA16_BASE_CH),
        '--checkpoints', str(na16_ckpt_dir),
        '--num_workers', '0',
        '--normal_loss_weight', str(NA16_NORMAL_WEIGHT),
        '--smooth_normal',
    ]
    print(f'>>> 16-bit normal-aware run: weight={NA16_NORMAL_WEIGHT}')
    print(f'>>> Checkpoint dir: {na16_ckpt_dir}')
    print('Running:', ' '.join(cmd))
    print('-' * 60)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    print(f'\n3.4 exit code: {proc.returncode}')


### 3.5 Ablation: normal-aware with DSINE pseudo-GT

Fix for the 3DGS normal problem. Uses DSINE (CVPR 2024) pseudo-GT normals
with a **separate normal prediction head** (not derived from depth via Sobel).

Architecture: normal head branches from UNet decoder features (d1).
Loss = depth_L1 + 0.5×grad + w×(1 − cos(pred_normal, DSINE_GT)).

Budget: ~3 min DSINE preprocessing, ~7h training per config.


In [ ]:
# === 3.5a Generate DSINE pseudo-GT normals (auto-skip if already done) ===
import subprocess, sys
from pathlib import Path

# Skip-guard: check if outputs already exist for both splits
need_run = False
for split in ('train', 'val'):
    img_dir  = Path(f'data/dataset/{split}/image')
    norm_dir = Path(f'data/dataset/{split}/normal_dsine')
    if not img_dir.exists():
        print(f'  [{split}] no image dir, skip')
        continue
    n_img  = len(list(img_dir.glob('*.png'))) + len(list(img_dir.glob('*.jpg')))
    n_norm = len(list(norm_dir.glob('*.png'))) if norm_dir.exists() else 0
    print(f'  [{split}] image={n_img}  normal_dsine={n_norm}')
    if n_norm < n_img:
        need_run = True

if need_run:
    print('\nRunning DSINE pseudo-GT normal generation ...')
    # Smoke test on 20 first so you can eyeball output before committing the full run.
    # If you've already eyeballed once, comment out the smoke line and run only the full one.
    subprocess.run(
        [sys.executable, 'scripts/compute_dsine_normals.py', '--limit', '20'],
        check=True)
    print('\n[smoke test done] inspect data/dataset/val/normal_dsine/*.png now.')
    print('If they look clean (smooth skin, no pits), continue with the full run:')
    subprocess.run(
        [sys.executable, 'scripts/compute_dsine_normals.py'],
        check=True)
else:
    print('\n[skip] DSINE pseudo-GT normals already generated for all splits.')


### 3.5b Preview: DSINE pseudo-GT vs 3DGS-rendered normals

Visual confirmation that DSINE produces clean normals without per-splat pits.


In [ ]:
# === 3.5b Side-by-side: 3DGS normal vs DSINE pseudo-GT ===
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from PIL import Image

val_dir = Path('data/dataset/val')
samples = sorted(p.stem for p in (val_dir / 'image').glob('*.png'))[:3]

fig, axes = plt.subplots(len(samples), 3, figsize=(12, 4 * len(samples)))
if len(samples) == 1:
    axes = axes[None, :]

for row, name in enumerate(samples):
    rgb   = np.array(Image.open(val_dir / 'image' / f'{name}.png').convert('RGB'))
    n_gs  = np.array(Image.open(val_dir / 'normal' / f'{name}.png').convert('RGB'))
    ds_p  = val_dir / 'normal_dsine' / f'{name}.png'
    n_dsine = np.array(Image.open(ds_p).convert('RGB')) if ds_p.exists() else np.zeros_like(n_gs)

    axes[row, 0].imshow(rgb);    axes[row, 0].set_title(f'{name} — RGB')
    axes[row, 1].imshow(n_gs);   axes[row, 1].set_title('3DGS-rendered normal (noisy)')
    axes[row, 2].imshow(n_dsine); axes[row, 2].set_title('DSINE pseudo-GT (clean)')
    for ax in axes[row]:
        ax.axis('off')
plt.tight_layout()
plt.show()
print('OK if DSINE column is visibly smoother on skin than 3DGS column.')


### 3.5c Train: 8-bit + DSINE normal (separate head)

Full retrain from scratch (~7h). `predict_normal=True` enables the independent
normal head. Best checkpoint selected by val_l1 (not val_loss).
Checkpoint: `checkpoints/depth_upres_8bit_ch32_normal_w050_dsine/`.


In [ ]:
# === 3.5c Train: 8-bit LR + DSINE normal-aware, retrain from scratch ===
import subprocess, sys
from pathlib import Path

ckpt_dir = Path('checkpoints/depth_upres_8bit_ch32_normal_w050_dsine')
done_marker = ckpt_dir / 'best.pt'

if done_marker.exists():
    print(f'[skip] final checkpoint exists: {done_marker}')
    print('Delete it (or rename ckpt_dir) to force retrain.')
else:
    cmd = [
        sys.executable, 'scripts/train_depth_upres.py',
        '--lr_input', '8bit',
        '--base_ch',  '32',
        '--batch_size', '2',
        '--grad_accum', '4',
        '--epochs', '100',
        '--amp',
        '--normal_loss_weight', '0.05',
        '--normal_dir_name', 'normal_dsine',
        '--checkpoints', str(ckpt_dir),
        '--num_workers', '0',
    ]
    print('CMD:', ' '.join(cmd))
    print('Expected wall time: ~3-4h on RTX 4070 Laptop (AMP, effective batch=8).')
    print('-' * 60)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    print(f'\n3.5c exit code: {proc.returncode}')


### 3.5d Train: 16-bit + DSINE normal (separate head)

Completes the 2×2 ablation grid:

|  | Depth-only | + DSINE normal |
|--|-----------|----------------|
| 8-bit | §3.1 | §3.5c |
| 16-bit | §3.2 | **§3.5d** |

Checkpoint: `checkpoints/depth_upres_16bit_ch32_normal_w050_dsine/`.


In [ ]:
# === 3.5d Train: 16-bit LR + DSINE normal-aware, retrain from scratch ===
import subprocess, sys
from pathlib import Path

ckpt_dir = Path('checkpoints/depth_upres_16bit_ch32_normal_w050_dsine')
done_marker = ckpt_dir / 'best.pt'

if done_marker.exists():
    print(f'[skip] final checkpoint exists: {done_marker}')
    print('Delete it (or rename ckpt_dir) to force retrain.')
else:
    cmd = [
        sys.executable, 'scripts/train_depth_upres.py',
        '--lr_input', '16bit',
        '--base_ch',  '32',
        '--batch_size', '2',
        '--grad_accum', '4',
        '--epochs', '100',
        '--amp',
        '--normal_loss_weight', '0.05',
        '--normal_dir_name', 'normal_dsine',
        '--checkpoints', str(ckpt_dir),
        '--num_workers', '0',
    ]
    print('CMD:', ' '.join(cmd))
    print('Expected wall time: ~3-4h on RTX 4070 Laptop (AMP, effective batch=8).')
    print('-' * 60)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    print(f'\n3.5d exit code: {proc.returncode}')


## Step 4. Evaluation

### 4.1 Training curves (auto-compare all runs)

Scans `checkpoints/depth_upres_*_ch*/train_log.json` and plots
train/val loss + val L1 across all runs.


In [ ]:
# === Plot training curves (auto-detect all runs and compare) ===
# Auto-discover all runs from checkpoints/depth_upres_*_ch*/train_log.json,
# plot train/val loss + val_l1 side by side for comparing 8bit vs 16bit / ch32 vs ch64.
import glob

log_paths = sorted(glob.glob('checkpoints/depth_upres_*_ch*/train_log.json'))
# Fall back to earliest single-path checkpoint
if not log_paths and Path('checkpoints/depth_upres/train_log.json').exists():
    log_paths = ['checkpoints/depth_upres/train_log.json']

if not log_paths:
    print('No training log yet.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    summary = []
    colors = plt.cm.tab10.colors
    for i, lp in enumerate(log_paths):
        history = json.loads(Path(lp).read_text())
        if not history:
            continue
        label = Path(lp).parent.name.replace('depth_upres_', '')
        # Shorten labels for readability
        label = label.replace('_ch32', '').replace('_normal_w050_dsine', '+N')
        label = label.replace('_ch64', '(ch64)')
        epochs     = [h['epoch']      for h in history]
        train_loss = [h['train_loss'] for h in history]
        val_loss   = [h['val_loss']   for h in history]
        val_l1     = [h['val_l1']     for h in history]
        col = colors[i % len(colors)]

        axes[0].plot(epochs, train_loss, label=f'{label} train', lw=1.6, ls='--', color=col)
        axes[0].plot(epochs, val_loss,   label=f'{label} val',   lw=2.0, ls='-',  color=col)
        # Raw (transparent) + smoothed (solid) for readability
        axes[1].plot(epochs, val_l1, alpha=0.2, lw=1, color=col)
        # Moving average (window=5)
        _w = min(5, len(val_l1))
        _smooth = np.convolve(val_l1, np.ones(_w)/_w, mode='valid')
        _ep_smooth = epochs[_w-1:]
        axes[1].plot(_ep_smooth, _smooth, label=label, lw=2.0, color=col)

        best_i = int(np.argmin(val_loss))
        summary.append({
            'run'        : label,
            'epochs'     : len(history),
            'best_epoch' : epochs[best_i],
            'best_val'   : float(val_loss[best_i]),
            'best_val_l1': float(val_l1[best_i]),
            'final_lr'   : float(history[-1]['lr']),
        })
        print(f"[{label}] {len(history)} ep, best val={val_loss[best_i]:.5f} @ epoch {epochs[best_i]}, val_l1={val_l1[best_i]:.5f}")

    axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss')
    axes[0].set_title('Train / Val Loss'); axes[0].legend(fontsize=7, loc='upper right'); axes[0].grid(alpha=0.3)
    axes[0].set_xlim(0, 105)
    axes[0].set_ylim(0, 0.02)  # zoom in on converged region
    axes[1].set_xlabel('epoch'); axes[1].set_ylabel('val L1')
    axes[1].set_title('Validation L1 (depth error)'); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
    axes[1].set_xlim(0, 105)
    axes[1].set_ylim(0.0018, 0.0035)  # show full converged range

    # Third subplot: best val_l1 bar chart per run
    if summary:
        names = [s['run'] for s in summary]
        vals  = [s['best_val_l1'] for s in summary]
        bars  = axes[2].bar(names, vals, color=colors[:len(names)])
        axes[2].set_ylabel('best val L1'); axes[2].set_title('Best val L1 per run')
        if vals: axes[2].set_ylim(min(vals)*0.95, max(vals)*1.08)  # zoom in on differences
        axes[2].grid(axis='y', alpha=0.3)
        for b, v in zip(bars, vals):
            axes[2].text(b.get_x() + b.get_width()/2, v, f'{v:.5f}',
                         ha='center', va='bottom', fontsize=8)
        plt.setp(axes[2].get_xticklabels(), rotation=20, ha='right')
    plt.tight_layout()
    plt.show()


### 4.2 Val inference visualization

3 random val samples: LR 8-bit | Bicubic ×4 | UNet prediction | HR ground truth.
Reproducible with seed=0.


In [ ]:
# === Run inference on val samples and visualize ===
import torch
import random
sys.path.insert(0, 'scripts')
from train_depth_upres import DepthUpResUNet
random.seed(0)   # reproducible viz pick

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Prefer newest ablation-style ckpt; fall back to old single-dir baseline.
VIZ_LR_KIND = '8bit'    # match cell 13 default; change to '16bit' to viz ablation
VIZ_BASE_CH = 64
_candidates = [
    Path(f'checkpoints/depth_upres_{VIZ_LR_KIND}_ch{VIZ_BASE_CH}/best.pt'),
    Path(f'checkpoints/depth_upres_{VIZ_LR_KIND}_ch32/best.pt'),
    Path('checkpoints/depth_upres/best.pt'),   # legacy 568-sample baseline
]
ckpt_path = next((p for p in _candidates if p.exists()), None)
if ckpt_path is None:
    print('No checkpoint yet.')
else:
    print(f'Using ckpt: {ckpt_path}')
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model = DepthUpResUNet(base_ch=ckpt['args'].get('base_ch', 32)).to(device)
    model.load_state_dict(ckpt['model'])
    model.eval()
    print(f"Loaded epoch {ckpt['epoch']}, val={ckpt['val_loss']:.5f}")

    val_lr = Path('data/dataset/val/depth_lr_8bit')
    val_hr = Path('data/dataset/val/depth')
    val_samples = random.sample(sorted(val_lr.glob('*.png')), 3)

    fig, axes = plt.subplots(3, 4, figsize=(18, 13))
    for row, p in enumerate(val_samples):
        name = p.name
        lr = cv2.imread(str(val_lr/name), cv2.IMREAD_UNCHANGED).astype(np.float32) / 255.0
        hr = cv2.imread(str(val_hr/name), cv2.IMREAD_UNCHANGED).astype(np.float32) / 65535.0

        with torch.no_grad():
            lr_t = torch.from_numpy(lr).unsqueeze(0).unsqueeze(0).to(device)
            pred = model(lr_t).cpu().squeeze().numpy()

        bicubic = cv2.resize(lr, (1024,1024), interpolation=cv2.INTER_CUBIC)

        axes[row,0].imshow(lr, cmap='inferno', vmin=0, vmax=1); axes[row,0].set_title(f'LR 256 8-bit\n{p.stem}')
        axes[row,1].imshow(bicubic, cmap='inferno', vmin=0, vmax=1); axes[row,1].set_title('Bicubic (baseline)')
        axes[row,2].imshow(pred, cmap='inferno', vmin=0, vmax=1); axes[row,2].set_title('UNet prediction')
        axes[row,3].imshow(hr, cmap='inferno', vmin=0, vmax=1); axes[row,3].set_title('HR ground truth')
        for ax in axes[row]: ax.axis('off')

        l1_bicubic = np.abs(bicubic - hr).mean()
        l1_pred = np.abs(pred - hr).mean()
        print(f'{p.stem}: bicubic L1={l1_bicubic:.4f}, UNet L1={l1_pred:.4f}, improvement={(l1_bicubic-l1_pred)/l1_bicubic*100:.1f}%')
    plt.tight_layout()
    plt.show()

### 4.3 Bicubic baseline vs UNet — full val set + error heatmaps

Quantitative: L1 + PSNR over entire val set.
Qualitative: 3 samples with error heatmaps
(|bicubic − GT|, |UNet − GT|, difference; red = UNet is better).


In [ ]:
# === Bicubic baseline vs UNet (parameterized eval, supports ablation comparison) ===
import torch, torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

sys.path.insert(0, 'scripts')
from train_depth_upres import DepthUpResUNet, DepthUpResDataset

# -------- Toggle: keep consistent with Cell 13 --------
EVAL_LR_KIND = '8bit'      # change to '16bit' to evaluate ablation model
EVAL_BASE_CH = 64          # match cell 13 default; falls back to ch32 if that was trained
# -------------------------------------------

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Prefer new parameterized checkpoint, fall back to legacy checkpoints/depth_upres/
candidates = [
    Path(f'checkpoints/depth_upres_{EVAL_LR_KIND}_ch{EVAL_BASE_CH}/best.pt'),
    Path(f'checkpoints/depth_upres_{EVAL_LR_KIND}_ch32/best.pt'),   # ch32 fallback
    Path('checkpoints/depth_upres/best.pt'),                        # legacy baseline
]
ckpt_path = next((p for p in candidates if p.exists()), None)
assert ckpt_path is not None, f'No checkpoint found in: {candidates}'

ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
base_ch = ckpt['args'].get('base_ch', 32)
model = DepthUpResUNet(base_ch=base_ch).to(device)
model.load_state_dict(ckpt['model'])
model.eval()
print(f"[ok] loaded {ckpt_path}")
print(f"     epoch={ckpt['epoch']}  val_loss={ckpt['val_loss']:.5f}  base_ch={base_ch}")
print(f"     LR_KIND for eval: {EVAL_LR_KIND}")

# ---- Full val set evaluation ---------------------------------------------------------
val_ds = DepthUpResDataset(Path('data/dataset'), 'val', lr_kind=EVAL_LR_KIND)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)
print(f'val samples: {len(val_ds)}')

def psnr(a, b, maxv=1.0):
    mse = F.mse_loss(a, b).item()
    return 20 * np.log10(maxv) - 10 * np.log10(mse + 1e-12)

bic_l1s, unet_l1s, bic_ps, unet_ps = [], [], [], []
with torch.no_grad():
    for lr_in, hr, *_ in val_loader:
        lr_in = lr_in.to(device); hr = hr.to(device)
        bic  = F.interpolate(lr_in, size=hr.shape[-2:], mode='bicubic', align_corners=False).clamp(0, 1)
        pred = model(lr_in).clamp(0, 1)
        bic_l1s.append(F.l1_loss(bic,  hr).item())
        unet_l1s.append(F.l1_loss(pred, hr).item())
        bic_ps.append(psnr(bic,  hr))
        unet_ps.append(psnr(pred, hr))

b_l1, u_l1 = float(np.mean(bic_l1s)), float(np.mean(unet_l1s))
b_ps, u_ps = float(np.mean(bic_ps)),  float(np.mean(unet_ps))
improve_pct = (b_l1 - u_l1) / b_l1 * 100

print('=' * 58)
print(f"Results for LR_KIND={EVAL_LR_KIND}  base_ch={base_ch}")
print(f"{'method':<12}{'val L1':>16}{'PSNR (dB)':>16}")
print('-' * 58)
print(f"{'bicubic':<12}{b_l1:>16.6f}{b_ps:>16.3f}")
print(f"{'UNet':<12}{u_l1:>16.6f}{u_ps:>16.3f}")
print('-' * 58)
print(f"improvement : L1 {improve_pct:+.2f}%   "
      f"(Δ={b_l1 - u_l1:+.6f}, ΔPSNR={u_ps - b_ps:+.3f} dB)")
print('=' * 58)

# ---- Error heatmaps: 3 samples ----------------------------------------------------
N_SHOW = 3
rng = np.random.default_rng(0)
idxs = rng.choice(len(val_ds), size=N_SHOW, replace=False).tolist()

fig, axes = plt.subplots(N_SHOW, 5, figsize=(22, 4.4 * N_SHOW))
with torch.no_grad():
    for row, idx in enumerate(idxs):
        lr_in, hr, *_ = val_ds[idx]
        name = val_ds.samples[idx]
        lr_in_b = lr_in.unsqueeze(0).to(device)
        hr_b    = hr.unsqueeze(0).to(device)
        bic  = F.interpolate(lr_in_b, size=hr_b.shape[-2:], mode='bicubic', align_corners=False).clamp(0, 1)
        pred = model(lr_in_b).clamp(0, 1)

        hr_np   = hr_b[0, 0].cpu().numpy()
        bic_np  = bic[0, 0].cpu().numpy()
        pred_np = pred[0, 0].cpu().numpy()

        d_bic  = np.abs(bic_np  - hr_np)
        d_unet = np.abs(pred_np - hr_np)
        vmax = max(d_bic.max(), d_unet.max()) + 1e-9
        diff_of_diff = d_bic - d_unet
        dd_lim = max(abs(diff_of_diff.min()), abs(diff_of_diff.max())) + 1e-9

        axes[row,0].imshow(hr_np, cmap='inferno', vmin=0, vmax=1)
        axes[row,0].set_title(f'HR ground truth\n{name}')
        axes[row,1].imshow(d_bic, cmap='hot', vmin=0, vmax=vmax)
        axes[row,1].set_title(f'|bicubic - GT|\nmean={d_bic.mean():.5f}')
        axes[row,2].imshow(d_unet, cmap='hot', vmin=0, vmax=vmax)
        axes[row,2].set_title(f'|UNet - GT|\nmean={d_unet.mean():.5f}')
        im3 = axes[row,3].imshow(np.clip(diff_of_diff, 0, None), cmap='hot', vmin=0, vmax=dd_lim)
        axes[row,3].set_title('bicubic_err - unet_err\n(red = UNet better)')
        axes[row,4].imshow(pred_np, cmap='inferno', vmin=0, vmax=1)
        imp = (d_bic.mean() - d_unet.mean()) / (d_bic.mean() + 1e-9) * 100
        axes[row,4].set_title(f'UNet prediction\nimprove={imp:+.1f}%')
        for ax in axes[row]: ax.axis('off')

plt.suptitle(f'Eval: LR_KIND={EVAL_LR_KIND}  |  base_ch={base_ch}', y=1.001)
plt.tight_layout()
out_png = ckpt_path.parent / f'eval_bicubic_vs_unet_{EVAL_LR_KIND}.png'
plt.savefig(out_png, dpi=120, bbox_inches='tight')
plt.show()
print(f'[ok] Error maps saved: {out_png}')


### 4.4 Numerical baseline table

Full-view averages: Nearest-8/16, Bicubic-8/16, all UNet variants, legacy.
Saved to `eval/baseline_table.csv`.


In [ ]:
# === 4.4 Final baseline table: all methods, L1 / RMSE / PSNR ===
import torch, torch.nn.functional as F
import numpy as np
import pandas as pd
from pathlib import Path
from torch.utils.data import DataLoader

sys.path.insert(0, 'scripts')
from train_depth_upres import DepthUpResUNet, DepthUpResDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Auto-discover checkpoints
def _find_ckpts():
    models = {}
    for bc in (64, 32):
        for kind in ('8bit', '16bit'):
            p = Path(f'checkpoints/depth_upres_{kind}_ch{bc}/best.pt')
            if p.exists(): models[f'UNet-{kind}'] = (p, kind)
            for wp in sorted(Path('checkpoints').glob(
                    f'depth_upres_{kind}_ch{bc}_normal_w*_dsine/best.pt')):
                models[f'UNet-{kind}+N'] = (wp, kind)
    lp = Path('checkpoints/depth_upres/best.pt')
    if lp.exists(): models['UNet-legacy'] = (lp, '8bit')
    return models

MODELS = _find_ckpts()
print(f'Models found: {list(MODELS.keys())}')

# Load datasets
val_ds_8  = DepthUpResDataset(Path('data/dataset'), 'val', lr_kind='8bit')
val_ds_16 = DepthUpResDataset(Path('data/dataset'), 'val', lr_kind='16bit')
N = len(val_ds_8)
print(f'val samples: {N}')

# Load models
loaded = {}
for mname, (ckpt_path, lr_kind) in MODELS.items():
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    bc = ckpt['args'].get('base_ch', 32)
    m = DepthUpResUNet(base_ch=bc).to(device).eval()
    m.load_state_dict(ckpt['model'], strict=False)
    loaded[mname] = (m, lr_kind)
    print(f"  {mname}: epoch={ckpt['epoch']}, base_ch={bc}")

def metrics(pred, hr):
    l1 = F.l1_loss(pred, hr).item()
    rmse = torch.sqrt(F.mse_loss(pred, hr)).item()
    mse = F.mse_loss(pred, hr).item()
    psnr = 20*np.log10(1.0) - 10*np.log10(mse + 1e-12)
    return l1, rmse, psnr

# Init result containers
rows = {'Nearest-8bit': [], 'Bicubic-8bit': [],
        'Nearest-16bit': [], 'Bicubic-16bit': []}
for tag in loaded: rows[tag] = []

with torch.no_grad():
    for idx in range(N):
        lr8, hr, *_ = val_ds_8[idx]
        lr16, *_ = val_ds_16[idx]
        lr8  = lr8.unsqueeze(0).to(device)
        lr16 = lr16.unsqueeze(0).to(device)
        hr   = hr.unsqueeze(0).to(device)

        near8 = F.interpolate(lr8, size=hr.shape[-2:], mode='nearest').clamp(0,1)
        bic8  = F.interpolate(lr8, size=hr.shape[-2:], mode='bicubic', align_corners=False).clamp(0,1)
        near16= F.interpolate(lr16, size=hr.shape[-2:], mode='nearest').clamp(0,1)
        bic16 = F.interpolate(lr16, size=hr.shape[-2:], mode='bicubic', align_corners=False).clamp(0,1)
        rows['Nearest-8bit'].append(metrics(near8, hr))
        rows['Bicubic-8bit'].append(metrics(bic8, hr))
        rows['Nearest-16bit'].append(metrics(near16, hr))
        rows['Bicubic-16bit'].append(metrics(bic16, hr))

        for tag, (m, lr_kind) in loaded.items():
            inp = lr16 if '16bit' in tag else lr8
            pred = m(inp)
            if isinstance(pred, tuple): pred = pred[0]
            pred = pred.clamp(0, 1)
            rows[tag].append(metrics(pred, hr))

# Build table
records = []
for name, vals in rows.items():
    if not vals: continue
    arr = np.array(vals)
    records.append({'method': name, 'L1': arr[:,0].mean(),
                    'RMSE': arr[:,1].mean(), 'PSNR_dB': arr[:,2].mean()})

df = pd.DataFrame(records).set_index('method')
order = ['Nearest-8bit','Bicubic-8bit','Nearest-16bit','Bicubic-16bit'] + list(MODELS.keys())
df = df.loc[[m for m in order if m in df.index]]

print('\n' + '='*66)
print('Full-view val baseline table')
print('='*66)
print(df.to_string(formatters={'L1':'{:.6f}'.format, 'RMSE':'{:.6f}'.format, 'PSNR_dB':'{:.2f}'.format}))

out_dir = Path('eval'); out_dir.mkdir(exist_ok=True)
df.to_csv(out_dir / 'baseline_table.csv')
print(f'\nSaved: eval/baseline_table.csv')


### 4.5 Multi-metric evaluation (PSNR / SSIM / RMSE / L1)

Adds SSIM on top of §4.4. Evaluates both full-view and zone-aware (face mask).
Saved to `eval/multimetric_{fullview,zoneaware}.csv`.


In [ ]:
# === 4.5 Multi-metric eval: PSNR / SSIM / RMSE / L1 (full-view) ===
import torch, torch.nn.functional as F
import numpy as np
import pandas as pd
from pathlib import Path
from torch.utils.data import DataLoader
from skimage.metrics import structural_similarity as ssim_fn

sys.path.insert(0, 'scripts')
from train_depth_upres import DepthUpResUNet, DepthUpResDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def compute_metrics(pred_np, gt_np):
    """pred_np, gt_np: (H,W) float32 in [0,1]."""
    l1 = np.abs(pred_np - gt_np).mean()
    rmse = np.sqrt(((pred_np - gt_np)**2).mean())
    mse = ((pred_np - gt_np)**2).mean()
    psnr = 20*np.log10(1.0) - 10*np.log10(mse + 1e-12)
    ss = ssim_fn(gt_np, pred_np, data_range=1.0)
    return {'L1': l1, 'RMSE': rmse, 'PSNR': psnr, 'SSIM': ss}

def compute_metrics_masked(pred_np, gt_np, mask_np):
    """Same but only within mask>0.5 pixels."""
    m = mask_np > 0.5
    if m.sum() < 100:
        return compute_metrics(pred_np, gt_np)  # fallback
    p, g = pred_np[m], gt_np[m]
    l1 = np.abs(p - g).mean()
    rmse = np.sqrt(((p - g)**2).mean())
    mse = ((p - g)**2).mean()
    psnr = 20*np.log10(1.0) - 10*np.log10(mse + 1e-12)
    # SSIM needs 2D, crop bounding box of mask
    ys, xs = np.where(m)
    y0, y1, x0, x1 = ys.min(), ys.max()+1, xs.min(), xs.max()+1
    crop_p = pred_np[y0:y1, x0:x1]
    crop_g = gt_np[y0:y1, x0:x1]
    ss = ssim_fn(crop_g, crop_p, data_range=1.0)
    return {'L1': l1, 'RMSE': rmse, 'PSNR': psnr, 'SSIM': ss}

# Auto-discover all checkpoints
def _find_ckpts():
    models = {}
    for bc in (64, 32):
        for kind in ('8bit', '16bit'):
            p = Path(f'checkpoints/depth_upres_{kind}_ch{bc}/best.pt')
            if p.exists(): models[f'UNet-{kind}-ch{bc}'] = (p, kind)
            # DSINE normal runs
            for wp in sorted(Path('checkpoints').glob(
                    f'depth_upres_{kind}_ch{bc}_normal_w*_dsine/best.pt')):
                tag = wp.parent.name.replace('depth_upres_', '')
                models[f'UNet-{tag}'] = (wp, kind)
    # Legacy
    lp = Path('checkpoints/depth_upres/best.pt')
    if lp.exists(): models['UNet-legacy'] = (lp, '8bit')
    return models

MODELS = _find_ckpts()
val_ds_8 = DepthUpResDataset(Path('data/dataset'), 'val', lr_kind='8bit')
val_ds_16 = DepthUpResDataset(Path('data/dataset'), 'val', lr_kind='16bit')
N = len(val_ds_8)
print(f'val samples: {N}')
print(f'Models found: {list(MODELS.keys())}')

# Load mask for zone-aware
mask_dir = Path('data/dataset/val/mask')

# Eval all
results_full = {}  # model_name -> list of metric dicts
results_zone = {}  # model_name -> list of metric dicts (masked)

# Interpolation baselines
for bname in ['Nearest-8bit', 'Bicubic-8bit', 'Nearest-16bit', 'Bicubic-16bit']:
    results_full[bname] = []
    results_zone[bname] = []
for mname in MODELS:
    results_full[mname] = []
    results_zone[mname] = []

# Load models
loaded = {}
for mname, (ckpt_path, lr_kind) in MODELS.items():
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    bc = ckpt['args'].get('base_ch', 32)
    m = DepthUpResUNet(base_ch=bc).to(device).eval()
    # strict=False to ignore normal_head weights when loading sep-head ckpt
    m.load_state_dict(ckpt['model'], strict=False)
    loaded[mname] = (m, lr_kind)
    print(f'  loaded {mname} (epoch={ckpt["epoch"]})')

with torch.no_grad():
    for idx in range(N):
        lr8, hr, *_ = val_ds_8[idx]
        lr16, *_ = val_ds_16[idx]
        name = val_ds_8.samples[idx]
        hr_np = hr[0].numpy()
        # Load mask
        mp = mask_dir / f'{name}.png'
        if mp.exists():
            mask_np = np.array(Image.open(mp).convert('L')).astype(np.float32) / 255.0
        else:
            mask_np = np.ones_like(hr_np)

        lr8_t = lr8.unsqueeze(0).to(device)
        lr16_t = lr16.unsqueeze(0).to(device)

        # Interpolation baselines
        near8 = F.interpolate(lr8_t, size=hr_np.shape, mode='nearest')[0,0].cpu().numpy().clip(0,1)
        bic8 = F.interpolate(lr8_t, size=hr_np.shape, mode='bicubic', align_corners=False)[0,0].cpu().numpy().clip(0,1)
        near16 = F.interpolate(lr16_t, size=hr_np.shape, mode='nearest')[0,0].cpu().numpy().clip(0,1)
        bic16 = F.interpolate(lr16_t, size=hr_np.shape, mode='bicubic', align_corners=False)[0,0].cpu().numpy().clip(0,1)

        for bname, pred_np in [('Nearest-8bit', near8), ('Bicubic-8bit', bic8),
                                ('Nearest-16bit', near16), ('Bicubic-16bit', bic16)]:
            results_full[bname].append(compute_metrics(pred_np, hr_np))
            results_zone[bname].append(compute_metrics_masked(pred_np, hr_np, mask_np))

        # Model predictions
        for mname, (model, lr_kind) in loaded.items():
            inp = lr16_t if '16bit' in mname else lr8_t
            out = model(inp)
            if isinstance(out, tuple): out = out[0]  # sep-head returns (depth, normal)
            pred_np = out[0,0].cpu().numpy().clip(0,1)
            results_full[mname].append(compute_metrics(pred_np, hr_np))
            results_zone[mname].append(compute_metrics_masked(pred_np, hr_np, mask_np))

# Build tables
def _build_table(results_dict):
    rows = []
    for name, metrics_list in results_dict.items():
        if not metrics_list: continue
        arr = {k: np.mean([m[k] for m in metrics_list]) for k in metrics_list[0]}
        arr['method'] = name
        rows.append(arr)
    df = pd.DataFrame(rows).set_index('method')
    return df[['L1', 'RMSE', 'PSNR', 'SSIM']]

df_full = _build_table(results_full)
df_zone = _build_table(results_zone)

print('\n' + '='*70)
print('FULL-VIEW evaluation (all pixels)')
print('='*70)
print(df_full.to_string(formatters={'L1':'{:.6f}'.format, 'RMSE':'{:.6f}'.format,
                                     'PSNR':'{:.2f}'.format, 'SSIM':'{:.4f}'.format}))

print('\n' + '='*70)
print('ZONE-AWARE evaluation (face mask only)')
print('='*70)
print(df_zone.to_string(formatters={'L1':'{:.6f}'.format, 'RMSE':'{:.6f}'.format,
                                     'PSNR':'{:.2f}'.format, 'SSIM':'{:.4f}'.format}))

# Save
out_dir = Path('eval'); out_dir.mkdir(exist_ok=True)
df_full.to_csv(out_dir / 'multimetric_fullview.csv')
df_zone.to_csv(out_dir / 'multimetric_zoneaware.csv')
print(f'\nSaved to eval/multimetric_fullview.csv and eval/multimetric_zoneaware.csv')


### 4.6 Visual comparison — zoom-in crops (paper figure)

3 val samples, 5 columns: LR 8-bit | Bicubic | UNet (ours) | GT | Error map.
Zoomed to nose/mouth region to highlight banding removal and edge sharpness.


In [ ]:
# === 4.6 Visual comparison: zoom-in crops for paper figure ===
import torch, cv2, random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

sys.path.insert(0, 'scripts')
from train_depth_upres import DepthUpResUNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(42)

# Load best 8-bit model
ckpt_candidates = [
    ('UNet+Normal', Path('checkpoints/depth_upres_8bit_ch32_normal_w050_dsine/best.pt')),
    ('UNet', Path('checkpoints/depth_upres_8bit_ch32/best.pt')),
]
ckpt_path = None
for label, p in ckpt_candidates:
    if p.exists():
        ckpt_path = p
        model_label = label
        break

if ckpt_path is None:
    print('No checkpoint found')
else:
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    bc = ckpt['args'].get('base_ch', 32)
    model = DepthUpResUNet(base_ch=bc).to(device).eval()
    model.load_state_dict(ckpt['model'], strict=False)
    print(f'Loaded: {ckpt_path.parent.name} (epoch {ckpt["epoch"]})')

    val_lr = Path('data/dataset/val/depth_lr_8bit')
    val_hr = Path('data/dataset/val/depth')
    val_mask = Path('data/dataset/val/mask')
    samples = random.sample(sorted(val_lr.glob('*.png')), 3)

    # Crop region: center 256x256 of 1024x1024 (nose/mouth area)
    C = 256
    s = (1024 - C) // 2

    fig, axes = plt.subplots(3, 6, figsize=(24, 12))
    for row, p in enumerate(samples):
        name = p.name
        lr = np.array(Image.open(val_lr/name), dtype=np.float32) / 255.0
        hr = np.array(Image.open(val_hr/name), dtype=np.float32) / 65535.0
        mask = np.array(Image.open(val_mask/name.replace('.png','.png')).convert('L'),
                       dtype=np.float32) / 255.0 if (val_mask/name).exists() else np.ones_like(hr)

        with torch.no_grad():
            lr_t = torch.from_numpy(lr).unsqueeze(0).unsqueeze(0).to(device)
            out = model(lr_t)
            if isinstance(out, tuple): out = out[0]
            pred = out.cpu().squeeze().numpy().clip(0,1)

        bic = cv2.resize(lr, (1024,1024), interpolation=cv2.INTER_CUBIC).clip(0,1)

        # Full view
        axes[row,0].imshow(lr, cmap='inferno', vmin=0, vmax=1, interpolation='nearest')
        axes[row,0].set_title(f'LR 256×256 8-bit\n{p.stem}', fontsize=9)

        # Zoom crops
        for col, (img, title) in enumerate([
            (bic[s:s+C, s:s+C], 'Bicubic'),
            (pred[s:s+C, s:s+C], f'{model_label} (ours)'),
            (hr[s:s+C, s:s+C], 'GT (1024 16-bit)'),
        ], start=1):
            axes[row,col].imshow(img, cmap='inferno', vmin=0, vmax=1, interpolation='nearest')
            axes[row,col].set_title(title, fontsize=9)

        # Error maps
        err_bic = np.abs(bic - hr)[s:s+C, s:s+C]
        err_pred = np.abs(pred - hr)[s:s+C, s:s+C]
        vmax = max(err_bic.max(), err_pred.max())
        axes[row,4].imshow(err_bic, cmap='hot', vmin=0, vmax=vmax)
        axes[row,4].set_title(f'|Bicubic-GT|\nmean={err_bic.mean():.5f}', fontsize=9)
        axes[row,5].imshow(err_pred, cmap='hot', vmin=0, vmax=vmax)
        axes[row,5].set_title(f'|Ours-GT|\nmean={err_pred.mean():.5f}', fontsize=9)

        for ax in axes[row]: ax.axis('off')

    plt.suptitle(f'Zoom-in comparison (center {C}×{C} crop)', y=1.01)
    plt.tight_layout()
    out_png = Path('eval/visual_comparison_zoomin.png')
    plt.savefig(out_png, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_png}')


### 4.6b Pixel-level banding artifact comparison (paper figure)

Shows 8-bit quantization banding (staircase artifacts) eliminated by SR.
Smooth regions (forehead/cheek) selected to maximize visibility.
Top row: 64×64 zoom crop. Bottom row: 1D depth profile across a scanline.


In [ ]:
# === 4.6b Pixel-level banding artifact comparison ===
import torch, cv2, random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

sys.path.insert(0, 'scripts')
from train_depth_upres import DepthUpResUNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(0)  # change seed to try different samples

# Load best 8-bit model
for p in [Path('checkpoints/depth_upres_8bit_ch32_normal_w050_dsine/best.pt'),
          Path('checkpoints/depth_upres_8bit_ch32/best.pt')]:
    if p.exists():
        ckpt = torch.load(p, map_location=device, weights_only=False)
        bc = ckpt['args'].get('base_ch', 32)
        model = DepthUpResUNet(base_ch=bc).to(device).eval()
        model.load_state_dict(ckpt['model'], strict=False)
        print(f'Loaded: {p.parent.name}')
        break

val_lr = Path('data/dataset/val/depth_lr_8bit')
val_hr = Path('data/dataset/val/depth')
samples = random.sample(sorted(val_lr.glob('*.png')), 2)

# Crop: forehead region (upper-center) where banding is most visible
# LR crop: 32x32 from (96, 80) -> HR crop: 128x128 from (384, 320)
# Crop: cheek/forehead area — smooth gradient where banding is obvious
# and UNet's dequantization advantage over bicubic is clearest
LR_CX, LR_CY, LR_CS = 90, 80, 40  # center_x, center_y, crop_size (in LR coords)
HR_CX, HR_CY, HR_CS = LR_CX*4, LR_CY*4, LR_CS*4
PROFILE_ROW_LR = LR_CY  # which row to plot as 1D profile
PROFILE_ROW_HR = HR_CY

fig, axes = plt.subplots(2*len(samples), 4, figsize=(16, 4*len(samples)),
                         gridspec_kw={'height_ratios': [3, 1]*len(samples)})

for si, p in enumerate(samples):
    name = p.name
    lr = np.array(Image.open(val_lr/name), dtype=np.float32) / 255.0
    hr = np.array(Image.open(val_hr/name), dtype=np.float32) / 65535.0

    with torch.no_grad():
        lr_t = torch.from_numpy(lr).unsqueeze(0).unsqueeze(0).to(device)
        out = model(lr_t)
        if isinstance(out, tuple): out = out[0]
        pred = out.cpu().squeeze().numpy().clip(0, 1)

    bic = cv2.resize(lr, (1024, 1024), interpolation=cv2.INTER_CUBIC).clip(0, 1)
    lr_nearest = cv2.resize(lr, (1024, 1024), interpolation=cv2.INTER_NEAREST)

    # Crops at HR resolution
    crops = {
        'LR nearest\n(8-bit banding)': lr_nearest[HR_CY:HR_CY+HR_CS, HR_CX:HR_CX+HR_CS],
        'Bicubic': bic[HR_CY:HR_CY+HR_CS, HR_CX:HR_CX+HR_CS],
        'Ours': pred[HR_CY:HR_CY+HR_CS, HR_CX:HR_CX+HR_CS],
        'GT (16-bit)': hr[HR_CY:HR_CY+HR_CS, HR_CX:HR_CX+HR_CS],
    }

    row_img = si * 2
    row_prof = si * 2 + 1

    # Shared vmin/vmax for this sample's crop
    all_vals = np.concatenate([c.ravel() for c in crops.values()])
    vmin, vmax = np.percentile(all_vals, [2, 98])

    for col, (title, crop) in enumerate(crops.items()):
        # Image crop with nearest interpolation (shows pixels)
        axes[row_img, col].imshow(crop, cmap='inferno', vmin=vmin, vmax=vmax,
                                  interpolation='nearest')
        axes[row_img, col].set_title(title, fontsize=10)
        axes[row_img, col].axis('off')
        if col == 0:
            axes[row_img, col].set_ylabel(p.stem, fontsize=9)

        # 1D depth profile (middle row of crop)
        mid = HR_CS // 2
        profile = crop[mid, :]
        axes[row_prof, col].plot(profile, color='orange', lw=1.5)
        axes[row_prof, col].set_ylim(vmin, vmax)
        axes[row_prof, col].set_xlim(0, len(profile))
        axes[row_prof, col].set_xticks([])
        if col == 0:
            axes[row_prof, col].set_ylabel('depth', fontsize=8)
            # Count unique values to show quantization
            n_unique = len(np.unique(np.round(profile * 255).astype(int)))
            axes[row_prof, col].set_title(f'{n_unique} unique values', fontsize=8)
        else:
            n_unique = len(np.unique(np.round(profile * 65535).astype(int)))
            axes[row_prof, col].set_title(f'{n_unique} unique values', fontsize=8)
        axes[row_prof, col].grid(alpha=0.3)

plt.suptitle('Pixel-level banding: 8-bit quantization creates staircase artifacts\n'
             'Top: zoomed crop (nearest interp) | Bottom: 1D depth profile along middle row',
             y=1.02, fontsize=11)
plt.tight_layout()
out_png = Path('eval/banding_comparison.png')
plt.savefig(out_png, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_png}')
print('\nLR nearest staircase artifacts are clearly visible in the 1D profile.')
print('Bicubic smooths the staircases but loses detail. Ours recovers smoothness while preserving GT detail.')


### 4.7 Normal head quality — angular error vs DSINE GT

Bonus output from the normal-aware model. Shows predicted normals vs DSINE GT
and per-pixel angular error maps. Confirms the model learns reasonable surface
normals as a side output.


In [ ]:
# === 4.7 Normal head quality: angular error vs DSINE GT ===
import torch, random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

sys.path.insert(0, 'scripts')
from train_depth_upres import DepthUpResUNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(42)

# Find a normal-aware checkpoint
ckpt_path = None
for p in sorted(Path('checkpoints').glob('depth_upres_*_normal_w*_dsine/best.pt')):
    ckpt_path = p
    break

if ckpt_path is None:
    print('No normal-aware checkpoint found')
else:
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    bc = ckpt['args'].get('base_ch', 32)
    model = DepthUpResUNet(base_ch=bc, predict_normal=True).to(device).eval()
    model.load_state_dict(ckpt['model'])
    lr_kind = ckpt['args'].get('lr_input', '8bit')
    print(f'Loaded: {ckpt_path.parent.name} (epoch {ckpt["epoch"]}, lr_kind={lr_kind})')

    val_lr = Path(f'data/dataset/val/depth_lr_{lr_kind}')
    val_dsine = Path('data/dataset/val/normal_dsine')
    val_mask = Path('data/dataset/val/mask')
    samples = random.sample(sorted(val_lr.glob('*.png')), 3)

    fig, axes = plt.subplots(3, 4, figsize=(18, 13))
    all_angles = []

    for row, p in enumerate(samples):
        name = p.stem
        # Load LR
        if lr_kind == '8bit':
            lr = np.array(Image.open(p), dtype=np.float32) / 255.0
        else:
            lr = np.array(Image.open(p), dtype=np.float32) / 65535.0

        # Load DSINE GT normal
        gt_n = np.array(Image.open(val_dsine / f'{name}.png').convert('RGB'), dtype=np.float32)
        gt_n = gt_n / 255.0 * 2.0 - 1.0  # [-1, 1]
        gt_n /= (np.linalg.norm(gt_n, axis=-1, keepdims=True) + 1e-8)

        # Load mask
        mp = val_mask / f'{name}.png'
        mask = np.array(Image.open(mp).convert('L'), dtype=np.float32) / 255.0 if mp.exists() else np.ones(gt_n.shape[:2])

        with torch.no_grad():
            lr_t = torch.from_numpy(lr).unsqueeze(0).unsqueeze(0).to(device)
            _, pred_n = model(lr_t)
            pred_n = pred_n[0].cpu().numpy()  # (3, H, W)
            pred_n = pred_n.transpose(1, 2, 0)  # (H, W, 3)

        # Angular error
        cos = np.clip((pred_n * gt_n).sum(axis=-1), -1, 1)
        angle = np.rad2deg(np.arccos(cos))

        m = mask > 0.5
        mean_angle = angle[m].mean() if m.any() else angle.mean()
        all_angles.append(mean_angle)

        # Viz: GT normal | Pred normal | Angular error | RGB overlay
        gt_rgb = ((gt_n + 1) * 0.5 * 255).clip(0, 255).astype(np.uint8)
        pred_rgb = ((pred_n + 1) * 0.5 * 255).clip(0, 255).astype(np.uint8)

        axes[row,0].imshow(gt_rgb)
        axes[row,0].set_title(f'DSINE GT normal\n{name}', fontsize=9)
        axes[row,1].imshow(pred_rgb)
        axes[row,1].set_title('Predicted normal\n(from normal head)', fontsize=9)
        im = axes[row,2].imshow(angle, cmap='jet', vmin=0, vmax=45)
        axes[row,2].set_title(f'Angular error (deg)\nmean(face)={mean_angle:.1f}°', fontsize=9)
        plt.colorbar(im, ax=axes[row,2], fraction=0.046)
        # Mask overlay
        angle_masked = angle.copy()
        angle_masked[~m] = 0
        axes[row,3].imshow(angle_masked, cmap='jet', vmin=0, vmax=45)
        axes[row,3].set_title(f'Angular error (face only)\nmean={mean_angle:.1f}°', fontsize=9)

        for ax in axes[row]: ax.axis('off')

    plt.suptitle(f'Normal head quality — mean angular error: {np.mean(all_angles):.1f}°', y=1.01)
    plt.tight_layout()
    out_png = Path('eval/normal_head_quality.png')
    plt.savefig(out_png, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Mean angular error (face region): {np.mean(all_angles):.1f}°')
    print(f'Saved: {out_png}')


### 4.8 SGNet baseline

Second external baseline (SwinIR-tiny already done). Tests whether RGB-guided
depth SR can outperform our depth-only approach on 3DGS degradation.


### 4.9 Downstream mesh quality (contribution d)

Core downstream evaluation: depth SR → backproject to point cloud → Chamfer distance vs GT.
Answers the question: "does depth SR actually produce better 3D geometry?"

Compares LR nearest / bicubic / UNet variants / GT.
Metrics: Chamfer-L1, Hausdorff, F-score.


In [ ]:
# === 4.9 Downstream mesh quality: depth → point cloud → Chamfer Distance ===
# Ref: Voynov et al. ICCV 2019, AIM 2024 Challenge
import subprocess

# Smoke test on 5 samples first
result = subprocess.run(
    [sys.executable, 'scripts/eval_mesh_quality.py', '--limit', '5'],
    capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
else:
    print('\n[smoke test OK] Running full eval...')
    result = subprocess.run(
        [sys.executable, 'scripts/eval_mesh_quality.py'],
        capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr)


### 4.10 Half-resolution eval (128→512) — fair SGNet comparison

SGNet runs at 128→512 (not 256→1024) due to 8GB VRAM constraint with RGB guidance.
This section evaluates all methods at half resolution for a fair comparison.


In [ ]:
# === 4.10 Half-res eval: 128->512, fair comparison with SGNet ===
import torch, torch.nn.functional as F
import numpy as np
import pandas as pd
import cv2
from pathlib import Path
from PIL import Image
from skimage.metrics import structural_similarity as ssim_fn

sys.path.insert(0, 'scripts')
from train_depth_upres import DepthUpResUNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

HR_SIZE = 512  # half of 1024
LR_SIZE = 128  # half of 256

def compute_metrics(pred_np, gt_np):
    l1 = np.abs(pred_np - gt_np).mean()
    rmse = np.sqrt(((pred_np - gt_np)**2).mean())
    mse = ((pred_np - gt_np)**2).mean()
    psnr = 20*np.log10(1.0) - 10*np.log10(mse + 1e-12)
    ss = ssim_fn(gt_np, pred_np, data_range=1.0)
    return {'L1': l1, 'RMSE': rmse, 'PSNR': psnr, 'SSIM': ss}

# Load UNet (8-bit best)
unet_models = {}
for label, p in [('UNet-8bit', Path('checkpoints/depth_upres_8bit_ch32/best.pt')),
                  ('UNet-8bit+N', Path('checkpoints/depth_upres_8bit_ch32_normal_w050_dsine/best.pt'))]:
    if p.exists():
        ckpt = torch.load(p, map_location=device, weights_only=False)
        bc = ckpt['args'].get('base_ch', 32)
        m = DepthUpResUNet(base_ch=bc).to(device).eval()
        m.load_state_dict(ckpt['model'], strict=False)
        unet_models[label] = m
        print(f'Loaded {label}: epoch={ckpt["epoch"]}')

# Load SGNet
sgnet_model = None
sgnet_path = Path('checkpoints/baseline_sgnet/best.pt')
if sgnet_path.exists():
    sgnet_dir = Path('external/SGNet')
    if str(sgnet_dir) not in sys.path:
        sys.path.insert(0, str(sgnet_dir))
    from models.SGNet import SGNet
    ckpt = torch.load(sgnet_path, map_location=device, weights_only=False)
    nf = ckpt['args'].get('num_feats', 24)
    sgnet_model = SGNet(num_feats=nf, kernel_size=3, scale=4).to(device).eval()
    sgnet_model.load_state_dict(ckpt['model'])
    print(f'Loaded SGNet: epoch={ckpt["epoch"]}, num_feats={nf}')
else:
    print('[WARN] SGNet checkpoint not found — will skip in table')

# Val data
val_hr_dir = Path('data/dataset/val/depth')
val_lr_dir = Path('data/dataset/val/depth_lr_8bit')
val_rgb_dir = Path('data/dataset/val/image')
samples = sorted(p.stem for p in val_hr_dir.glob('*.png'))
N = len(samples)
print(f'Val samples: {N}')

results = {'Nearest': [], 'Bicubic': []}
for k in unet_models: results[k] = []
if sgnet_model: results['SGNet'] = []

with torch.no_grad():
    for i, name in enumerate(samples):
        # Load and resize HR to 512
        hr_full = np.array(Image.open(val_hr_dir / f'{name}.png'), dtype=np.float32)
        hr_full = hr_full / 65535.0 if hr_full.max() > 255 else hr_full / 255.0
        hr = cv2.resize(hr_full, (HR_SIZE, HR_SIZE), interpolation=cv2.INTER_AREA)

        # Load LR and resize to 128
        lr_full = np.array(Image.open(val_lr_dir / f'{name}.png'), dtype=np.float32) / 255.0
        lr = cv2.resize(lr_full, (LR_SIZE, LR_SIZE), interpolation=cv2.INTER_AREA)

        # Interpolation baselines (128 -> 512)
        near = cv2.resize(lr, (HR_SIZE, HR_SIZE), interpolation=cv2.INTER_NEAREST)
        bic = cv2.resize(lr, (HR_SIZE, HR_SIZE), interpolation=cv2.INTER_CUBIC).clip(0, 1)
        results['Nearest'].append(compute_metrics(near, hr))
        results['Bicubic'].append(compute_metrics(bic, hr))

        # UNet: input 128, model does bicubic×4 internally -> 512
        lr_t = torch.from_numpy(lr).unsqueeze(0).unsqueeze(0).to(device)
        for label, model in unet_models.items():
            out = model(lr_t)
            if isinstance(out, tuple): out = out[0]
            pred = out[0, 0].cpu().numpy().clip(0, 1)
            results[label].append(compute_metrics(pred, hr))

        # SGNet: input (rgb_512, lr_128) -> 512
        if sgnet_model:
            rgb_img = Image.open(val_rgb_dir / f'{name}.png').convert('RGB')
            rgb_img = rgb_img.resize((HR_SIZE, HR_SIZE), Image.BICUBIC)
            rgb = np.array(rgb_img, dtype=np.float32) / 255.0
            rgb_t = torch.from_numpy(rgb.transpose(2, 0, 1)).unsqueeze(0).to(device)
            out, _ = sgnet_model((rgb_t, lr_t))
            pred = out[0, 0].cpu().numpy().clip(0, 1)
            results['SGNet'].append(compute_metrics(pred, hr))

        if (i+1) % 20 == 0:
            print(f'  [{i+1}/{N}]')

# Build table
rows = []
for name, metrics_list in results.items():
    if not metrics_list: continue
    arr = {k: np.mean([m[k] for m in metrics_list]) for k in metrics_list[0]}
    arr['method'] = name
    rows.append(arr)

df = pd.DataFrame(rows).set_index('method')
df = df[['L1', 'RMSE', 'PSNR', 'SSIM']]

print('\n' + '='*70)
print(f'HALF-RESOLUTION eval: {LR_SIZE}→{HR_SIZE} (×4)')
print(f'Fair comparison with SGNet (same input/output resolution)')
print('='*70)
print(df.to_string(formatters={'L1':'{:.6f}'.format, 'RMSE':'{:.6f}'.format,
                               'PSNR':'{:.2f}'.format, 'SSIM':'{:.4f}'.format}))

out_dir = Path('eval'); out_dir.mkdir(exist_ok=True)
df.to_csv(out_dir / 'halfres_comparison.csv')
print(f'\nSaved: eval/halfres_comparison.csv')
